# 1 — Measure: subsample splits, **hand-driven**This is the escape hatch. `1_Measure_Splits.ipynb` runs the recipe library through`splits.run()`; this notebook does the same chain **one comparison at a time, byhand**, so you can stop in the middle, plot an intermediate, or use a selection thatisn't worth writing a recipe for.**It runs exactly the same calls in exactly the same order** — that's the point.`splits._measure_side()` is this chain plus bookkeeping, and the acceptance test forthe runner is that a hand-run split and a runner-run split agree to the last digit.If they ever disagree, this notebook is how you find out where.```build_stacks(keep_cube=True)  -> measure_all_bins(seed=...)  -> fit_profile / plot_flux_profile_two(fit=True)  -> find_core_halo_boundary  -> measure_core_twohalo_velocity / measure_twohalo_luminosity  -> measure_onehalo_luminosity  (or the PSF-corrected pair at level='full')  -> measure_three_zone_ratios  -> bootstrap_fit_profile```**Use the batch notebook instead** for anything you'll run more than twice — itcaches per side, resumes after an interrupt, and records provenance. This one doesnone of that.

## 0 · Setup — same imports and config as the batch notebook

In [ ]:
import importlib
from dataclasses import replace

import numpy as np
import matplotlib.pyplot as plt

from run_config import cfg, smoke
from utils_lya_halo import (measure, PipelineConfig, plotting, analysis, multicat,
    selection, core, stack, fitting, formatting, splits, guide)
from utils_lya_halo.io import read_galaxy_fits, apply_finite_cut
from utils_lya_halo.stack import build_stacks

LYA_REST = PipelineConfig.LYA_REST

import warnings
from astropy.utils.exceptions import AstropyUserWarning
warnings.filterwarnings("ignore", message="Input data contains invalid values.*",
                        category=AstropyUserWarning)

Config is duplicated here rather than imported so this notebook stands alone. **Keepit identical to step 1 of `1_Measure_Splits.ipynb`** — if the two drift, hand-run andrunner-run numbers stop being comparable and the whole reason for this notebook goesaway.

In [ ]:
random_seed  = 10
bin_mode     = 'kpc'
bins         = [0, 10, 16, 24, 36, 60, 100, 180, 350, 600, 1000, 2000, 3000]
half_width   = 3.0
stack_method = 'biweight'

cfg2 = replace(cfg,
    field='BOTH', bin_mode=bin_mode, bins=bins, mask_method="segmap",
    bg_inner_arcsec=80, bg_outer_arcsec=90,
    rest_wave_min=1100, rest_wave_max=1700, n_bootstrap=2000,
    line_window=[LYA_REST - half_width, LYA_REST + half_width])

np.random.seed(random_seed)

# The fit settings. Same values as FIT_KW in the batch notebook -- if you change one
# here the two notebooks' numbers are no longer comparable.
GAMMA_FIXED = 0.8
R_C_FIXED   = 400.0

PRODUCT_PATHS = {
    'AEGIS':  './outputs/galaxy_cat65a007_b12_992952_AEGIS_MOSDEF_kpc_biweight_segmap_bg80-90.fits',
    'COSMOS': './outputs/galaxy_cat65a007_b12_992952_COSMOS_MOSDEF_kpc_biweight_segmap_bg80-90.fits',
}
products = {
    label: apply_finite_cut(read_galaxy_fits(path), cfg2.cut_radial_bin, cfg2.min_good_wave)
    for label, path in PRODUCT_PATHS.items()
}
prod_combined = multicat.combine_products(products, label_col="SUBSET")
print(f"COMBINED: {prod_combined.ngal} galaxies")

## 1 · Choose the two samples**This is the only cell you edit.** Everything below runs unchanged whatever `g`contains, as long as it is a dict of two products.Three ways to build it — uncomment one:

In [ ]:
# --- (a) percentile split on a derived quantity ------------------------------
g = selection.split_by_derived(prod_combined, "SFR_50 / (np.pi*(RE_BEST**2))",
                               name="sigma_SFR", percentiles=50,
                               labels=["0-50%", "50-100%"])

# --- (b) a hand-written boolean mask ------------------------------------------
# my_mask = (prod_combined.catalog["MASS_50"] > 10.0)
# g = selection.split_by_mask(prod_combined, my_mask, labels=["low M*", "high M*"])

# --- (c) reuse a library recipe's own cut, without the runner ------------------
# rec  = splits.LIBRARY["inclination"]
# el   = splits.resolve_eligibility(prod_combined.catalog, rec)
# cuts = splits.cut(prod_combined, rec, eligible=el["mask"])
# g    = {lab: info["product"] for lab, info in cuts.items()}

labels = tuple(g.keys())
prod1, prod2 = g[labels[0]], g[labels[1]]
print(f"{labels[0]}: {prod1.ngal} galaxies")
print(f"{labels[1]}: {prod2.ngal} galaxies")
if min(prod1.ngal, prod2.ngal) < 80:
    print("\n** below the library's default min_n=80 -- fit parameters will be "
          "poorly constrained on the smaller side.")

## 2 · Stack`keep_cube=True` is **required**: the PSF-corrected core measurement and theper-galaxy rescale-and-recoadd both need the cube, and they fail late andconfusingly without it.

In [ ]:
stacks  = multicat.stack_each(cfg2, g, keep_cube=True)
stacks1, stacks2 = stacks[labels[0]], stacks[labels[1]]

### Sanity check: are the two samples matched in everything *except* the split variable?

In [ ]:
for col, fmt, scale in [("MASS_50", ".3f", 1), ("z", ".3f", 1), ("EBV", ".3f", 1),
                        ("SFR_50", ".3f", 1), ("HB_FLUX", ".3f", 1e17),
                        ("RE_BEST", ".3f", 1), ("AGE_50", ".3f", 1)]:
    try:
        a = np.nanmedian(prod1.catalog[col]) * scale
        b = np.nanmedian(prod2.catalog[col]) * scale
    except KeyError:
        print(f"{col:<10} (not in catalog)")
        continue
    print(f"{col:<10} {labels[0]:>12}: {a:{fmt}}    {labels[1]:>12}: {b:{fmt}}")

## 3 · Bootstrap the radial bins`seed=` is shared between the two sides deliberately. The sides contain **different**galaxies, so their bootstraps are still independent — sharing the seed just makes thecomparison reproducible and lets differences be taken draw-by-draw later, whichpreserves skew that quadrature would flatten.

In [ ]:
boots = multicat.measure_each(cfg2, stacks, seed=random_seed)
boot1, boot2 = boots[labels[0]], boots[labels[1]]

# equivalent, one at a time:
# boot1 = measure.measure_all_bins(cfg2, stacks1, seed=random_seed, verbose=True)
# boot2 = measure.measure_all_bins(cfg2, stacks2, seed=random_seed, verbose=True)

## 4 · Look at it before you measure it

In [ ]:
plotting.plot_stack_panels_two(stacks1, stacks2,
                               method=stack_method,
                               distance_type=bin_mode,
                               labels=list(labels),
                               layout="side",
                               xlim=(1180, 1300),
                               lya_window=half_width)

In [ ]:
xlims    = (3, 3000)
vr_ticks = (0.1, 0.2, 0.5, 1, 2, 5, 10, 20)

# --- centroid vs radius: the primary result ---
analysis.plot_centroid_profile_two(
    boot1, boot2, stacks1, stacks2,
    labels=list(labels),
    bin_mode="kpc", xlims=xlims, vr_ticks=vr_ticks,
    ylims=(-200, 400),
    colors=("tab:blue", "tab:red"),
    fmts=("o-", "s--"),
    markersize=6, linewidth=1.5, capsize=3.5,
    sample_b_alpha=1.0,
    jitter=0.05,              # x-offset so the two sets of error bars don't overlap
    figsize=(8, 4),
    legend_title="sample",
    save_fig=False, save_name="Centroid_Comparison",
)

## 5 · Fit the profile`fit=True` on the flux plot returns the fit results alongside the figure, so thenumbers you measure below come from **the same fit that is drawn**, not a second one.`method="psf"` forward-models the fiber PSF and therefore *uses* the innermost bin —that's the fiducial. `method="naive"` skips inner bins instead and exists only forthe head-to-head comparison.

In [ ]:
fig, ax, fit_a, fit_b = analysis.plot_flux_profile_two(
    boot1, boot2, stacks1, stacks2, labels=list(labels),
    bin_mode="kpc", xlims=xlims, vr_ticks=vr_ticks,
    logy=True, sample_b_alpha=1.0, colors=("tab:blue", "tab:red"), jitter=0.05,
    figsize=(9, 6), title=None,
    fit=True,                                    # exp-core + power-law overlay
    gamma_fixed=GAMMA_FIXED, r_c_fixed=R_C_FIXED,
    save_fig=False, save_name="Surface_Luminosity_Comparison")

for lab, fr in zip(labels, (fit_a, fit_b)):
    print(f"{lab:>12}  h1={fr.get('h1'):.2f}  success={fr.get('success')}")

## 6 · The core / two-halo boundary`find_core_halo_boundary` returns the radius where the exp-core term hands over tothe outer power law. **Check it before using it** — a fit whose core term neverdominates (a net-absorption or very-low-N stack) returns a finite but meaninglesscrossover at ~0 kpc, and everything downstream then silently measures a zone with nobins in it. That is the failure that killed the 2026-08-01 batch; the batch runnernow guards against it, and the guard is reproduced here.

In [ ]:
boundary1 = fitting.find_core_halo_boundary(fit_a)
boundary2 = fitting.find_core_halo_boundary(fit_b)

def check_boundary(binfo, fit, label):
    """Same test measure_onehalo_luminosity applies internally: is there at least
    one bin strictly outside the core bin whose midpoint falls inside the boundary?"""
    b = binfo.get("boundary_radius")
    if b is None or not np.isfinite(b):
        print(f"{label:>12}  NO BOUNDARY -- zone quantities are not measurable")
        return False
    re_ = np.asarray(fit["r_edges"], float)
    rmid = 0.5 * (re_[:-1] + re_[1:])
    ok = np.any((np.arange(len(rmid)) > 0) & (rmid <= float(b)))
    if not ok:
        print(f"{label:>12}  DEGENERATE r_x={b:.4g} kpc (inside r_edges[1]="
              f"{re_[1]:.4g}) -- the exp-core term never dominates. The expcore "
              f"model does not describe this sample; h1 and r_x are NOT measured.")
        return False
    src = "own fit" if binfo.get("boundary_from_own_fit") is not False else "FALLBACK fit"
    print(f"{label:>12}  r_x = {b:7.2f} kpc   (from the {src})")
    return True

ok1 = check_boundary(boundary1, fit_a, labels[0])
ok2 = check_boundary(boundary2, fit_b, labels[1])

## 7 · Zone quantitiesThree zones, partitioned by radius: **core** (innermost bin), **one-halo** (out to thecrossover), **two-halo** (beyond it).`level="full"` in the batch runner corresponds to `psf_corrected=True` here. It isrequired before quoting **any** core/one-halo ratio: without it the core luminosityisn't computed at all and the one-halo zone is not leak-corrected, so flux the PSFscattered out of the core gets double-counted.

In [ ]:
def zone_properties(product, stacks, boot, boundary, *, psf_corrected=True):
    """The 'full'-level chain. psf_corrected=False is the 'quick'/'standard'
    equivalent -- faster, but core_lum is None and the one-halo zone is NOT
    leak-corrected, so do not quote a core/one-halo ratio from it."""
    vel         = measure.measure_core_twohalo_velocity(boot, boundary)
    twohalo_lum = measure.measure_twohalo_luminosity(boot, boundary)
    if psf_corrected:
        core_lum, onehalo_lum = measure.measure_psf_corrected_core_onehalo_luminosity(
            cfg2, product, stacks, boot, boundary, seed=random_seed)
        ratios = measure.measure_three_zone_ratios(core_lum, onehalo_lum, twohalo_lum)
    else:
        core_lum, ratios = None, None
        onehalo_lum = measure.measure_onehalo_luminosity(boot, boundary)
    return dict(boundary_info=boundary, vel=vel, core_lum=core_lum,
                onehalo_lum=onehalo_lum, twohalo_lum=twohalo_lum, ratios=ratios)

props1 = zone_properties(prod1, stacks1, boot1, boundary1) if ok1 else None
props2 = zone_properties(prod2, stacks2, boot2, boundary2) if ok2 else None

## 8 · Error bars on the fit parameters themselves`bootstrap_fit_profile` refits the profile on every bootstrap draw. This is the**only** uncertainty on `h1` / `r_c` / `crossover_radius` this pipeline trusts —covariance-matrix errors are known unreliable for `r_c` and `gamma`, which aredegenerate with each other.It is expensive: each draw is a separate nonlinear fit. The batch runner uses 200;2000 is for numbers going into the paper.Note what this does **not** do: the zone split in step 7 still uses the fixedpoint-estimate boundary. That's the deliberate "cheap and honest" choice — a nestedper-draw re-partition would be defensible but much more expensive and harder toexplain.

In [ ]:
NBOOT_FIT = 2000        # 200 for a quick look, 2000 for the paper

boot_fit1 = analysis.bootstrap_fit_profile(
    boot1, fit_a["r_edges"], fit_a["r_fine"],
    model="expcore", method="psf", R=fit_a["R"],
    gamma_fixed=GAMMA_FIXED, r_c_fixed=R_C_FIXED,
    nboot=NBOOT_FIT, seed=random_seed, verbose=True)

boot_fit2 = analysis.bootstrap_fit_profile(
    boot2, fit_b["r_edges"], fit_b["r_fine"],
    model="expcore", method="psf", R=fit_b["R"],
    gamma_fixed=GAMMA_FIXED, r_c_fixed=R_C_FIXED,
    nboot=NBOOT_FIT, seed=random_seed, verbose=True)

for lab, bf in zip(labels, (boot_fit1, boot_fit2)):
    nf = bf.get("n_failed") or 0
    nu = (bf.get("meta") or {}).get("nboot_used") or 1
    flag = "  ** >10% failed: the 16/84 spread is built on survivors and is optimistic" if nf > 0.1 * nu else ""
    print(f"{lab:>12}  {nf}/{nu} refit draws failed{flag}")

## 9 · Print and tabulate

In [ ]:
analysis.describe_subsample_properties(
    label=labels[0], n_gal=prod1.ngal, base_sample="AGN-excluded",
    fit_result=fit_a, boot_fit=boot_fit1, **props1)
print()
analysis.describe_subsample_properties(
    label=labels[1], n_gal=prod2.ngal, base_sample="AGN-excluded",
    fit_result=fit_b, boot_fit=boot_fit2, **props2)

In [ ]:
subsamples = [
    {"label": labels[0], "boot_fit": boot_fit1, **props1},
    {"label": labels[1], "boot_fit": boot_fit2, **props2},
]

print(formatting.make_subsample_latex_table(
    subsamples,
    caption="Derived properties by subsample.",
    label="tab:subsample-properties",
    scales={"L_2halo": 1e43},
))

---## Appendix · Does the hand-run agree with the runner?The acceptance test for `splits.run()` is that these two paths give the same number.If you've changed anything in `splits.py`, this is how you check you didn't break it.`check_cut_equivalence` covers the cutting half; comparing `h1` covers themeasurement half.

In [ ]:
# rec = splits.LIBRARY["mass_half"]
# splits.check_cut_equivalence(prod_combined, rec, id_col="ID", verbose=True)
#
# res = splits.run(cfg2, prod_combined, rec, splitcat=sc, level="standard",
#                  seed=random_seed, nboot_fit=NBOOT_FIT,
#                  fit_kw=dict(gamma_fixed=GAMMA_FIXED, r_c_fixed=R_C_FIXED))
# for lab, side in res["mass_half"]["sides"].items():
#     print(lab, side["fit_result"]["h1"])       # should match fit_a/fit_b above